In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.payments"
)

silver_df = spark.table(
    "customer360_dev.silver.payments"
)

reject_df = spark.table(
    "customer360_dev.quarantine.payment_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 6,000,150
Silver rows   : 5,652,000
Rejected rows : 348,150


In [0]:
orders_df = (
    spark.table("customer360_dev.silver.orders")
    .select("order_id")
)

customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

validated_df = (
    bronze_df
    .withColumn(
        "payment_attempt_no_cast",
        F.col("payment_attempt_no").cast("int")
    )
    .withColumn(
        "payment_amount_cast",
        F.col("payment_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "refund_amount_cast",
        F.col("refund_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "payment_timestamp_cast",
        F.to_timestamp("payment_timestamp")
    )
    .withColumn(
        "payment_status_clean",
        F.upper(F.trim("payment_status"))
    )
    .join(
        orders_df.withColumn(
            "_order_exists",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_payment_id",
        F.col("payment_id").isNull()
        | (F.trim(F.col("payment_id")) == "")
    )
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_order_not_found",
        F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_invalid_attempt_no",
        F.col("payment_attempt_no_cast").isNull()
        | (F.col("payment_attempt_no_cast") <= 0)
    )
    .withColumn(
        "dq_invalid_payment_amount",
        F.col("payment_amount_cast").isNull()
        | (F.col("payment_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_refund_amount",
        F.col("refund_amount_cast").isNotNull()
        & (F.col("refund_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_payment_timestamp",
        F.col("payment_timestamp_cast").isNull()
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("payment_status_clean").isin(
            "INITIATED",
            "SUCCESS",
            "FAILED",
            "REFUNDED",
            "PARTIALLY_REFUNDED"
        )
    )
    .withColumn(
        "dq_refund_exceeds_payment",
        F.col("refund_amount_cast").isNotNull()
        & (
            F.col("refund_amount_cast")
            > F.col("payment_amount_cast")
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_payment_id")
    | F.col("dq_missing_order_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_order_not_found")
    | F.col("dq_customer_not_found")
    | F.col("dq_invalid_attempt_no")
    | F.col("dq_invalid_payment_amount")
    | F.col("dq_invalid_refund_amount")
    | F.col("dq_invalid_payment_timestamp")
    | F.col("dq_invalid_status")
    | F.col("dq_refund_exceeds_payment")
)

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Valid candidates before dedup: 5,652,000
Duplicate rows removed: 0


In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 6,000,150
Silver rows        : 5,652,000
Rejected rows      : 348,150
Duplicates removed : 0
Accounted for      : 6,000,150
Difference         : 0


In [0]:
result = [
    {
        "entity": "payments",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
6000150,6000150,0,0,payments,348150,5652000,PASS


In [0]:
orphan_orders = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.orders"
        ).select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

print(
    "Payments with missing Silver Order:",
    orphan_orders
)

Payments with missing Silver Order: 0


In [0]:
orphan_customers = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        ).select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Payments with missing Silver Customer:",
    orphan_customers
)

Payments with missing Silver Customer: 0


In [0]:
warning_summary = (
    silver_df
    .select(
        F.sum(
            F.when(
                F.col(
                    "dq_warning_refund_status_without_amount"
                ),
                1
            ).otherwise(0)
        ).alias("refund_status_without_amount"),

        F.sum(
            F.when(
                F.col(
                    "dq_warning_refund_without_timestamp"
                ),
                1
            ).otherwise(0)
        ).alias("refund_without_timestamp"),

        F.sum(
            F.when(
                F.col(
                    "dq_warning_duplicate_success_txn_ref"
                ),
                1
            ).otherwise(0)
        ).alias("duplicate_success_transaction_refs")
    )
)

display(warning_summary)

refund_status_without_amount,refund_without_timestamp,duplicate_success_transaction_refs
0,0,0


In [0]:
(
    reject_df
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+---------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|rejection_reason                                                                                                                                         |count |
+---------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|ORDER_NOT_FOUND                                                                                                                                          |240000|
|ORDER_NOT_FOUND; CUSTOMER_NOT_FOUND                                                                                                                      |84000 |
|INVALID_PAYMENT_STATUS                                                                                                                                   |12000 |
|REFUND_EXCEEDS_PAYMEN